In [4]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries Included Successfully")

Libraries Included Successfully


In [5]:
# =========================================================
# 1. LOAD DATA
# =========================================================
df = pd.read_csv("uber_data_cleaned.csv")

In [6]:
print("Dataset loaded successfully!")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Dataset loaded successfully!
Shape: (99359, 26)

Columns:
['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'trip_duration_minutes', 'pickup_hour', 'pickup_day', 'pickup_weekday', 'payment_type_name', 'fare_per_mile', 'total_per_mile']


In [8]:
# =========================================================
# 2. BASIC CLEANING
# =========================================================
# Remove completely duplicate rows if any
df.drop_duplicates(inplace=True)

In [9]:
# Remove rows where target is missing
df = df.dropna(subset=["total_amount"])

In [10]:
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,...,tolls_amount,improvement_surcharge,total_amount,trip_duration_minutes,pickup_hour,pickup_day,pickup_weekday,payment_type_name,fare_per_mile,total_per_mile
0,1,2016-03-01 00:00:00,2016-03-01 00:07:55,1,2.50,-73.976746,40.765152,1,N,-74.004265,...,0.00,0.3,12.35,7.916667,0,1,Tuesday,Credit card,3.600000,4.940000
1,1,2016-03-01 00:00:00,2016-03-01 00:11:06,1,2.90,-73.983482,40.767925,1,N,-74.005943,...,0.00,0.3,15.35,11.100000,0,1,Tuesday,Credit card,3.793103,5.293103
2,2,2016-03-01 00:00:00,2016-03-01 00:31:06,2,19.98,-73.782021,40.644810,1,N,-73.974541,...,0.00,0.3,63.80,31.100000,0,1,Tuesday,Credit card,2.727728,3.193193
3,1,2016-03-01 00:00:01,2016-03-01 00:16:04,1,6.20,-73.788773,40.647758,1,N,-73.829208,...,0.00,0.3,21.80,16.050000,0,1,Tuesday,No charge,3.306452,3.516129
4,1,2016-03-01 00:00:01,2016-03-01 00:05:00,1,0.70,-73.958221,40.764641,1,N,-73.967896,...,0.00,0.3,8.80,4.983333,0,1,Tuesday,Credit card,7.857143,12.571429
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99354,1,2016-03-01 06:17:10,2016-03-01 06:22:15,1,0.50,-73.990898,40.750519,1,N,-73.998245,...,0.00,0.3,5.80,5.083333,6,1,Tuesday,Cash,10.000000,11.600000
99355,1,2016-03-01 06:17:10,2016-03-01 06:32:41,1,3.40,-74.014488,40.718296,1,N,-73.982361,...,0.00,0.3,16.80,15.516667,6,1,Tuesday,Credit card,4.117647,4.941176
99356,1,2016-03-01 06:17:10,2016-03-01 06:37:23,1,9.70,-73.963379,40.774097,1,N,-73.865028,...,5.54,0.3,44.14,20.216667,6,1,Tuesday,Credit card,2.989691,4.550515
99357,2,2016-03-01 06:17:10,2016-03-01 06:22:09,1,0.92,-73.984901,40.763111,1,N,-73.970695,...,0.00,0.3,8.16,4.983333,6,1,Tuesday,Credit card,5.978261,8.869565


In [11]:
# Optional: remove impossible/invalid rows
df = df[df["trip_distance"] > 0]
df = df[df["trip_duration_minutes"] > 0]
df = df[df["total_amount"] > 0]

print("\nAfter basic cleaning, shape:", df.shape)


After basic cleaning, shape: (99357, 26)


In [14]:
# =========================================================
# 3. REMOVE UNNECESSARY / LEAKAGE COLUMNS
# =========================================================
# We are predicting total_amount
target_column = "total_amount"

# Columns to drop because they are:
# - datetime raw columns (already feature engineered)
# - target leakage columns
# - duplicate engineered text columns
columns_to_drop = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "payment_type_name",
    "fare_amount",              # leakage
    "extra",                    # leakage
    "mta_tax",                  # leakage
    "tip_amount",               # leakage
    "tolls_amount",             # leakage
    "improvement_surcharge",    # leakage
    "fare_per_mile",            # derived from fare_amount
    "total_per_mile",           # derived from total_amount (direct leakage)
    "payment_type"              # optional removal to reduce leakage from tip/payment behavior
]

# Drop only columns that actually exist in dataset
columns_to_drop = [col for col in columns_to_drop if col in df.columns]
columns_to_drop

['tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'payment_type_name',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'fare_per_mile',
 'total_per_mile',
 'payment_type']

In [15]:
df_model = df.drop(columns=columns_to_drop)

print("\nColumns used after dropping unnecessary ones:")
print(df_model.columns.tolist())


Columns used after dropping unnecessary ones:
['VendorID', 'passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude', 'RatecodeID', 'store_and_fwd_flag', 'dropoff_longitude', 'dropoff_latitude', 'total_amount', 'trip_duration_minutes', 'pickup_hour', 'pickup_day', 'pickup_weekday']


In [20]:
df_model["store_and_fwd_flag"].unique()

<StringArray>
['N', 'Y']
Length: 2, dtype: str

In [24]:
df_model["pickup_weekday"].unique()

<StringArray>
['Tuesday', 'Thursday']
Length: 2, dtype: str

In [25]:
# Convert store_and_fwd_flag (Y/N) into numeric
if "store_and_fwd_flag" in df_model.columns:
    df_model["store_and_fwd_flag"] = df_model["store_and_fwd_flag"].map({"N": 0, "Y": 1})

df_model


,VendorID,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,total_amount,trip_duration_minutes,pickup_hour,pickup_day,pickup_weekday
0,1,1,2.50,-73.976746,40.765152,1,0,-74.004265,40.746128,12.35,7.916667,0,1,Tuesday
1,1,1,2.90,-73.983482,40.767925,1,0,-74.005943,40.733166,15.35,11.100000,0,1,Tuesday
2,2,2,19.98,-73.782021,40.644810,1,0,-73.974541,40.675770,63.80,31.100000,0,1,Tuesday
3,1,1,6.20,-73.788773,40.647758,1,0,-73.829208,40.712345,21.80,16.050000,0,1,Tuesday
4,1,1,0.70,-73.958221,40.764641,1,0,-73.967896,40.762901,8.80,4.983333,0,1,Tuesday
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99354,1,1,0.50,-73.990898,40.750519,1,0,-73.998245,40.750462,5.80,5.083333,6,1,Tuesday
99355,1,1,3.40,-74.014488,40.718296,1,0,-73.982361,40.752529,16.80,15.516667,6,1,Tuesday
99356,1,1,9.70,-73.963379,40.774097,1,0,-73.865028,40.770512,44.14,20.216667,6,1,Tuesday
99357,2,1,0.92,-73.984901,40.763111,1,0,-73.970695,40.759148,8.16,4.983333,6,1,Tuesday


In [26]:
# Convert pickup_weekday text to numeric if needed
if "pickup_weekday" in df_model.columns:
    weekday_mapping = {
        "Monday": 0,
        "Tuesday": 1,
        "Wednesday": 2,
        "Thursday": 3,
        "Friday": 4,
        "Saturday": 5,
        "Sunday": 6
    }
    df_model["pickup_weekday"] = df_model["pickup_weekday"].map(weekday_mapping)
df_model

,VendorID,passenger_count,trip_distance,pickup_longitude,pickup_latitude,RatecodeID,store_and_fwd_flag,dropoff_longitude,dropoff_latitude,total_amount,trip_duration_minutes,pickup_hour,pickup_day,pickup_weekday
0,1,1,2.50,-73.976746,40.765152,1,0,-74.004265,40.746128,12.35,7.916667,0,1,1
1,1,1,2.90,-73.983482,40.767925,1,0,-74.005943,40.733166,15.35,11.100000,0,1,1
2,2,2,19.98,-73.782021,40.644810,1,0,-73.974541,40.675770,63.80,31.100000,0,1,1
3,1,1,6.20,-73.788773,40.647758,1,0,-73.829208,40.712345,21.80,16.050000,0,1,1
4,1,1,0.70,-73.958221,40.764641,1,0,-73.967896,40.762901,8.80,4.983333,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99354,1,1,0.50,-73.990898,40.750519,1,0,-73.998245,40.750462,5.80,5.083333,6,1,1
99355,1,1,3.40,-74.014488,40.718296,1,0,-73.982361,40.752529,16.80,15.516667,6,1,1
99356,1,1,9.70,-73.963379,40.774097,1,0,-73.865028,40.770512,44.14,20.216667,6,1,1
99357,2,1,0.92,-73.984901,40.763111,1,0,-73.970695,40.759148,8.16,4.983333,6,1,1


In [27]:

df_model.isna().sum()
#

VendorID                 0
passenger_count          0
trip_distance            0
pickup_longitude         0
pickup_latitude          0
RatecodeID               0
store_and_fwd_flag       0
dropoff_longitude        0
dropoff_latitude         0
total_amount             0
trip_duration_minutes    0
pickup_hour              0
pickup_day               0
pickup_weekday           0
dtype: int64

In [28]:
# Fill any remaining missing values
df_model = df_model.fillna(df_model.median(numeric_only=True))

# If any object columns still remain, remove them (safe fallback)
object_cols = df_model.select_dtypes(include=["object"]).columns.tolist()

object_cols

[]

In [29]:
df_model.info()

<class 'pandas.DataFrame'>
Index: 99357 entries, 0 to 99358
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   VendorID               99357 non-null  int64  
 1   passenger_count        99357 non-null  int64  
 2   trip_distance          99357 non-null  float64
 3   pickup_longitude       99357 non-null  float64
 4   pickup_latitude        99357 non-null  float64
 5   RatecodeID             99357 non-null  int64  
 6   store_and_fwd_flag     99357 non-null  int64  
 7   dropoff_longitude      99357 non-null  float64
 8   dropoff_latitude       99357 non-null  float64
 9   total_amount           99357 non-null  float64
 10  trip_duration_minutes  99357 non-null  float64
 11  pickup_hour            99357 non-null  int64  
 12  pickup_day             99357 non-null  int64  
 13  pickup_weekday         99357 non-null  int64  
dtypes: float64(7), int64(7)
memory usage: 11.4 MB


In [30]:
if object_cols:
    print("\nRemoving remaining object columns:", object_cols)
    df_model = df_model.drop(columns=object_cols)
#the above code will not work bcz object_cols remains empty